# Claude Haiku 5.5 — ZERO-SHOT with explicit density definition Experiment A (reference only) — FINAL RUN

Closed-model replication of the open-source experiments. Prompt wording is identical to the Qwen / LLaVA / Gemma (and GPT) notebooks of the same condition.

* Model: `claude-haiku-5-5` via the **Messages API**, adaptive thinking, effort **medium**, images resized with `thumbnail((512, 512))` exactly like the open-source runs.
* The pilot is a separate notebook (`Haiku_ExperimentA_ZeroShot_DensityDef_Pilot.ipynb`); this notebook is the full 1600-image run.
* Full run = **Message Batches API** (50% cheaper). Batch ids are saved to Drive, so a Colab disconnect loses nothing.
* Add `ANTHROPIC_API_KEY` in Colab **Secrets** (key icon). Never paste it into a cell.

## 1. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

## 2. Back up any existing Haiku Experiment A results before overwriting

In [ ]:
import os, shutil

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

for fname in ["haiku_experiment_A_zeroshot_densitydef.csv", "haiku_experiment_A_zeroshot_densitydef_evaluated.csv"]:
    src = os.path.join(RESULT_DIR, fname)
    if os.path.exists(src):
        backup = os.path.join(RESULT_DIR, fname.replace(".csv", "_backup.csv"))
        if not os.path.exists(backup):
            shutil.copy2(src, backup)
            print(f"Backed up {fname} -> {os.path.basename(backup)}")
        else:
            print(f"Backup already exists for {fname}, skipping.")
    else:
        print(f"No existing {fname} found — nothing to back up.")

## 3. Install packages

In [ ]:
!pip install -q -U anthropic

## 4. Connect to Claude Haiku 5.5 & configuration
Everything that affects the results is set here. `temperature` / `top_p` / `top_k` are **not** sent (Claude Haiku 5.5 rejects non-default sampling parameters), so runs are not strictly deterministic. Thinking is adaptive and **effort is `medium`** (the model's default).
Prices are Anthropic's published Batch prices for Claude Haiku 5.5 (prompts up to 100K tokens) — check them on Anthropic's pricing page. `QUEUE_REQUEST_LIMIT` is the number of batch requests that may wait in the queue at once (Start tier 200,000 / Build 300,000 / Scale 500,000), so it never binds for a 1,600-request experiment.

In [ ]:
import os, io, re, json, time, math, base64, random, hashlib
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=5)

COND = "zsdd_A"
MODEL      = "claude-haiku-5-5"
EFFORT     = "medium"            # low | medium | high | xhigh | max   (Haiku 5.5 default = medium)
MAX_TOKENS = 16000               # cap incl. thinking tokens (not a cost)

PRICE_IN, PRICE_OUT = 0.10, 0.50 # USD per 1M tokens (standard price, prompts <= 100K tokens) -- VERIFY
BATCH_DISCOUNT = 0.5             # Message Batches API = 50% of standard prices

MAX_BATCH_MB        = 200        # hard limit is 256 MB per batch
QUEUE_REQUEST_LIMIT = 200_000    # batch requests allowed in the processing queue at once (see the note above)

CONFIG = dict(model=MODEL, effort=EFFORT, thinking="adaptive", max_tokens=MAX_TOKENS, image_max=512, cond=COND)
CONFIG_HASH = hashlib.sha1(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:10]
print(CONFIG, CONFIG_HASH)

## 5. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Same resize as the open-source runs."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

@lru_cache(maxsize=3000)
def img_b64(path):
    buf = io.BytesIO()
    load_image(path).save(buf, format="JPEG", quality=95)
    return base64.standard_b64encode(buf.getvalue()).decode()

def _img(path):
    return {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64(path)}}

## 6. Zero-shot prompt builder — Experiment A
System prompt is the zero-shot version of the open-source notebooks (with the explicit Low/Medium/High definition). The builder returns one Messages API request body (used for both the pilot and the Batch file).

In [ ]:
SYSTEM_PROMPT = """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels — judge by how visually packed the crowd appears, not by exact count:

0 = Low: People are clearly separated, with visible space between individuals. Each person is easy to distinguish on their own.

1 = Medium: People are moderately close together. Some overlap between individuals, but gaps and open space are still visible.

2 = High: People are tightly packed with little or no visible space between them. Individuals often overlap and may be hard to distinguish separately.

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""

TEST_TURN = """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""


def _params(system, messages):
    """One Messages-API request body. It is used as-is for the pilot and as `params` of a Batch request."""
    return {"model": MODEL, "max_tokens": MAX_TOKENS, "system": system, "messages": messages,
            "thinking": {"type": "adaptive"}, "output_config": {"effort": EFFORT}}


def build_messages_zeroshot_A(test_image):
    """Messages-API request body for one test image — zero-shot Experiment A."""
    msgs = [{"role": "user", "content": [_img(test_image), {"type": "text", "text": TEST_TURN}]}]
    return _params(SYSTEM_PROMPT, msgs)


## 7. Inference and parsing (same robust parser used throughout this project)

In [ ]:
def run_claude(request, retries=3):
    """One synchronous call (pilot / 9c re-run only). Returns (message_dict, error_string)."""
    last = None
    for attempt in range(retries):
        try:
            return client.messages.create(**request).model_dump(mode="json"), None
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            time.sleep(2 ** attempt)
    return None, last


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than taking them literally."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density

def extract_text(msg):
    """Answer text = all `text` blocks. `thinking` blocks (empty by default) are ignored."""
    if not msg:
        return ""
    return "".join(b.get("text", "") for b in (msg.get("content") or []) if b.get("type") == "text").strip()


def usage_of(msg):
    """in_tok = input tokens; out_tok = output tokens INCLUDING thinking (the API does not split them)."""
    u = (msg or {}).get("usage") or {}
    return dict(in_tok=(u.get("input_tokens") or 0) + (u.get("cache_creation_input_tokens") or 0) + (u.get("cache_read_input_tokens") or 0),
                out_tok=u.get("output_tokens") or 0)


def row_from_message(fname, msg, err=None):
    """One result row from a Messages-API message dict (used for the pilot and when assembling the batch results)."""
    text = extract_text(msg)
    pc, pd_ = parse_response(text)
    stop = (msg or {}).get("stop_reason")
    return dict(test_filename=int(fname), predicted_count=pc, predicted_density=pd_, raw_response=text,
                status=stop, incomplete=(stop if stop not in (None, "end_turn", "stop_sequence") else None),
                error=err, **usage_of(msg))


# offline self-test (no API call)
_fake = {"stop_reason": "end_turn", "content": [{"type": "thinking", "thinking": "", "signature": "x"},
         {"type": "text", "text": "Count: 123\n\nDensity: 2"}],
         "usage": {"input_tokens": 900, "output_tokens": 400}}
assert parse_response(extract_text(_fake)) == (123, 2)
assert parse_response("Count: 12345678\nDensity: 1") == (None, 1)
assert usage_of(_fake) == dict(in_tok=900, out_tok=400)
assert row_from_message(7, _fake)["incomplete"] is None
assert row_from_message(7, dict(_fake, stop_reason="max_tokens"))["incomplete"] == "max_tokens"
print("Parser self-test passed.")

## 8. Load data & checks (no API calls)
No `support_mapping.csv` needed for zero-shot. Prints the rendered prompt structure so you can confirm it before spending anything.

In [ ]:
import pandas as pd
import os

ROOT = "/content/Phase5_Input"
TEST_DIR = f"{ROOT}/test/images"

test_df = pd.read_csv(f"{ROOT}/test.csv")

TEST_FILES = sorted(test_df["filename"].astype(int).tolist())

def test_path(f): return os.path.join(TEST_DIR, f"{int(f):04d}.jpg")

def make_request(f):
    f = int(f)
    return build_messages_zeroshot_A(test_path(f))

# ---- data checks ----
assert len(TEST_FILES) == len(set(TEST_FILES)) == 1600, len(TEST_FILES)
_missing = [test_path(f) for f in TEST_FILES if not os.path.exists(test_path(f))]
assert not _missing, f"{len(_missing)} image files missing, e.g. {_missing[:3]}"
print("Data checks passed:", len(TEST_FILES), "test images")

In [ ]:
def render(req):
    out = ["[system] " + req["system"].split("\n")[0] + " ..."]
    for m in req["messages"]:
        if isinstance(m["content"], str):
            out.append(f"[{m['role']}] {m['content']!r}")
        else:
            parts = ["<image>" if c["type"] == "image" else c["text"] for c in m["content"]]
            out.append(f"[{m['role']}] " + " | ".join(repr(p) for p in parts))
    return "\n".join(out)

_req = make_request(TEST_FILES[0])
_n_img = sum(1 for m in _req["messages"] if not isinstance(m["content"], str) for c in m["content"] if c["type"] == "image")
assert _n_img == 1, _n_img
assert not ({"temperature", "top_p", "top_k", "stream"} & set(_req))
assert _req["thinking"] == {"type": "adaptive"} and _req["output_config"] == {"effort": EFFORT}
assert _req["messages"][-1]["role"] == "user" and all(m["role"] in ("user", "assistant") for m in _req["messages"])
assert ("YOLO predicted count: " in render(_req)) == False
print(render(_req))
print("\nImages in request:", _n_img, "| request size:", round(len(json.dumps(_req)) / 1024), "KB")

## 9. Full run — all 1600 test images via the Message Batches API, saved to Drive
**Spends real money** (Claude Haiku 5.5 is cheap: well under a few dollars per condition at Batch prices, see the pilot's projection). Two steps:
* **9a** builds the batch file(s) (each under 200 MB; the API limit is 256 MB) and submits them. Gated by `CONFIRM_FULL_RUN`. Re-run it any time: it only submits what is still pending.
* **9b** (run in this or any later session) checks status, downloads the results and writes `haiku_experiment_A_zeroshot_densitydef.csv` to Drive. Batch ids live in a state file on Drive; results stay on Anthropic's servers for 29 days.

### 9a-1. Batch helpers and files (run this cell EVERY session, including when you come back to collect results)
It only defines the helpers, loads the saved batch state and builds the batch file(s) if they do not exist yet (a few minutes). It never submits anything.

In [ ]:
STATE_PATH = f"{RESULT_DIR}/haiku_batch_state_{COND}_{CONFIG_HASH}.json"
BATCH_LOCAL = f"/content/haiku_batch_{COND}"
RAW_DIR = f"{RESULT_DIR}/haiku_raw_batch_{COND}_{CONFIG_HASH}"
os.makedirs(BATCH_LOCAL, exist_ok=True); os.makedirs(RAW_DIR, exist_ok=True)   # processing_status: in_progress | canceling | ended

def load_state():
    return json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else None

def save_state(st):
    json.dump(st, open(STATE_PATH + ".tmp", "w"), indent=1); os.replace(STATE_PATH + ".tmp", STATE_PATH)

def build_batch_files():
    old = load_state(); old_by = {e["file"]: e for e in old["entries"]} if old else {}
    entries, part, lines, size = [], 0, [], 0
    def flush():
        nonlocal part, lines, size
        if not lines: return
        part += 1; name = f"{COND}_part{part:03d}.jsonl"; path = f"{BATCH_LOCAL}/{name}"
        open(path, "w", encoding="utf-8").write("".join(lines))
        entries.append(dict(file=name, path=path, n=len(lines), bytes=size, batch_id=None, status=None, counts=None, downloaded=False))
        lines, size = [], 0
    for f in TEST_FILES:
        line = json.dumps({"custom_id": f"{COND}__{int(f):04d}", "params": make_request(f)}) + "\n"
        if lines and size + len(line.encode()) > MAX_BATCH_MB * 1024 * 1024: flush()
        lines.append(line); size += len(line.encode())
    flush()
    for e in entries:                       # keep submission progress of identical files
        o = old_by.get(e["file"])
        if o and o["n"] == e["n"]:
            for k in ("batch_id", "status", "counts", "downloaded"): e[k] = o[k]
    st = dict(config=CONFIG, entries=entries); save_state(st); return st

def refresh_states(st):
    for e in st["entries"]:
        if e["batch_id"] and e["status"] != "ended":
            b = client.messages.batches.retrieve(e["batch_id"])
            e["status"], e["counts"] = b.processing_status, b.request_counts.model_dump()
    save_state(st)

st = load_state()
if st is None or not all(os.path.exists(e["path"]) for e in st["entries"] if not e["batch_id"]):
    t0 = time.time(); st = build_batch_files(); print(f"Built batch files in {time.time() - t0:.0f}s")
assert sum(e["n"] for e in st["entries"]) == len(TEST_FILES), "request count mismatch"
print("Files:", [(e["file"], e["n"], round(e["bytes"] / 1e6, 1)) for e in st["entries"]])

### 9a-2. Submit (run only when you are ready to spend money)
Set `CONFIRM_FULL_RUN = True` to submit. Safe to re-run: parts that already have a batch id are skipped, and a batch that ended without a single success is reset and submitted again. If it says a batch file is missing (Colab was reset), run 9a-1 first. When you come back later just to collect results, skip this cell and go to 9b.

In [ ]:
CONFIRM_FULL_RUN = False          # <-- set True to submit

def _read_requests(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

if CONFIRM_FULL_RUN:
    refresh_states(st)
    for e in st["entries"]:        # a batch that ended without a single success (all expired / errored / canceled) is submitted again
        c = e.get("counts") or {}
        if e["batch_id"] and e["status"] == "ended" and not c.get("succeeded"):
            print("resetting", e["file"], c, "-> will be resubmitted")
            e.update(batch_id=None, status=None, counts=None, downloaded=False)
    save_state(st)
    in_flight = sum(e["n"] for e in st["entries"] if e["batch_id"] and e["status"] != "ended")
    for e in st["entries"]:
        if e["batch_id"]: continue
        if not os.path.exists(e["path"]):
            print("batch file missing (Colab was reset): run 9a-1 first, then this cell again."); break
        if in_flight + e["n"] > QUEUE_REQUEST_LIMIT:
            print(f"Queue budget reached ({in_flight:,} requests in flight). Re-run this cell later."); break
        try:
            b = client.messages.batches.create(requests=_read_requests(e["path"]))
        except anthropic.APIStatusError as ex:
            print("could not submit", e["file"], "->", ex.status_code, str(ex.message)[:300], "\nRe-run this cell later."); break
        e.update(batch_id=b.id, status=b.processing_status, counts=b.request_counts.model_dump())
        save_state(st)                      # persist the batch id IMMEDIATELY
        in_flight += e["n"]
        print("submitted", e["file"], "->", b.id, b.processing_status)
else:
    print("CONFIRM_FULL_RUN is False -> nothing submitted.")

In [ ]:
# 9b — status, download, assemble  (safe to re-run; works in a new session too)
OUT_PATH = os.path.join(RESULT_DIR, "haiku_experiment_A_zeroshot_densitydef.csv")
st = load_state(); refresh_states(st)

for e in st["entries"]:
    if e["batch_id"] and e["status"] == "ended" and not e["downloaded"]:
        with open(f"{RAW_DIR}/{e['file']}.out.jsonl", "w", encoding="utf-8") as fh:
            for r in client.messages.batches.results(e["batch_id"]):
                fh.write(json.dumps(r.model_dump(mode="json")) + "\n")
        e["downloaded"] = True; save_state(st)

for e in st["entries"]:
    print(e["file"], "|", e["batch_id"], "|", e["status"], "|", e.get("counts"), "| downloaded:", e["downloaded"])

got = {}
for p in sorted(os.listdir(RAW_DIR), key=lambda x: (x.startswith("sync_fixes"), x)):   # sync fixes applied last
    if p.endswith(".out.jsonl"):
        for line in open(f"{RAW_DIR}/{p}", encoding="utf-8"):
            if line.strip():
                d = json.loads(line); got[d["custom_id"]] = d      # matched by custom_id: result order is not guaranteed

rows, missing = [], []
for f in TEST_FILES:
    d = got.get(f"{COND}__{int(f):04d}")
    if d is None: missing.append(f); continue
    r = d["result"]
    if r["type"] == "succeeded":
        rows.append(row_from_message(f, r["message"]))
    elif r["type"] == "errored":
        rows.append(row_from_message(f, None, err=json.dumps(r.get("error"))[:300]))
    else:                                   # canceled / expired: no answer
        missing.append(f)

results = pd.DataFrame(rows)
if len(results):
    results.to_csv(OUT_PATH, index=False)
    batch_cost = (results.in_tok.sum() * PRICE_IN + results.out_tok.sum() * PRICE_OUT) / 1e6 * BATCH_DISCOUNT
    print(f"\nAssembled {len(results)}/{len(TEST_FILES)} rows -> {OUT_PATH}  (actual cost at Batch prices: ${batch_cost:.2f})")
if missing:
    print(f"WAITING / MISSING: {len(missing)} results not available yet, e.g. {missing[:5]}. Re-run this cell when the batches finish.")
failed = int(results.predicted_count.isna().sum()) if len(results) else 0
print("Unparseable counts:", failed)
if len(results):
    print("Stop reasons:", results.status.value_counts().to_dict(), "| errored:", int(results.error.notna().sum()))

### 9c. Optional - re-run missing, errored or incomplete requests synchronously
Only needed if some batch requests expired or errored (or stopped at `max_tokens`). Gated; results are merged by `custom_id` the next time 9b runs. Sync calls cost twice the Batch price (still cents for Claude Haiku 5.5).

In [ ]:
RERUN_MISSING = False             # True = re-run missing / errored / incomplete requests synchronously (2x the Batch price)

if RERUN_MISSING:
    bad = set(missing) | (set(results.loc[results.error.notna() | results.incomplete.notna(), "test_filename"]) if len(results) else set())
    print("To re-run:", len(bad))

    def _fix(f):
        msg, err = run_claude(make_request(f))
        return int(f), msg, err

    with open(f"{RAW_DIR}/sync_fixes.out.jsonl", "a", encoding="utf-8") as out, ThreadPoolExecutor(4) as ex:
        for f, msg, err in ex.map(_fix, sorted(bad)):
            if msg:
                out.write(json.dumps({"custom_id": f"{COND}__{f:04d}", "result": {"type": "succeeded", "message": msg}}) + "\n")
    print("Done -> re-run the 9b cell above to re-assemble.")
else:
    print("RERUN_MISSING is False -> nothing re-run.")

## 10. Evaluate — counting metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

assert len(results) == len(TEST_FILES), f"Only {len(results)}/{len(TEST_FILES)} results are in - finish step 9b first."
haiku_A_zsdd = results.copy()

haiku_A_zsdd_eval = haiku_A_zsdd.merge(
    test_df[["filename", "total_count", "crowd_density"]],
    left_on="test_filename",
    right_on="filename",
    how="left"
)

print("Missing ground truth after merge:", haiku_A_zsdd_eval["total_count"].isna().sum())

n_before = len(haiku_A_zsdd_eval)
count_valid = haiku_A_zsdd_eval.dropna(subset=["predicted_count"]).copy()
n_dropped = n_before - len(count_valid)
print(f"Dropped {n_dropped}/{n_before} rows with no parseable count ({100*n_dropped/n_before:.1f}%)")

y_true_count = count_valid["total_count"]
y_pred_count = count_valid["predicted_count"]

mae = mean_absolute_error(y_true_count, y_pred_count)
rmse = np.sqrt(mean_squared_error(y_true_count, y_pred_count))

nonzero = count_valid["total_count"] > 0
mape = np.mean(np.abs(
    (count_valid.loc[nonzero, "total_count"] - count_valid.loc[nonzero, "predicted_count"])
    / count_valid.loc[nonzero, "total_count"]
)) * 100

print("Haiku Experiment A (Zero-Shot, density definition) — Counting")
print("=" * 44)
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"(computed on {len(count_valid)}/{n_before} images with a parseable count)")

## 11. Evaluate — density classification

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

density_valid = haiku_A_zsdd_eval.dropna(subset=["predicted_density"]).copy()
n_density_dropped = len(haiku_A_zsdd_eval) - len(density_valid)
print(f"Dropped {n_density_dropped}/{len(haiku_A_zsdd_eval)} rows with no parseable density ({100*n_density_dropped/len(haiku_A_zsdd_eval):.1f}%)")

y_true_density = density_valid["crowd_density"]
y_pred_density = density_valid["predicted_density"]

accuracy = accuracy_score(y_true_density, y_pred_density)
precision = precision_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
recall = recall_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
f1 = f1_score(y_true_density, y_pred_density, average="weighted", zero_division=0)

print("Haiku Experiment A (Zero-Shot, density definition) — Density")
print("=" * 44)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"(trivial 'always High' baseline: {(test_df['crowd_density'] == 2).mean():.4f})")

cm = confusion_matrix(y_true_density, y_pred_density, labels=[0, 1, 2])
print()
print("Confusion matrix (rows=GT, cols=pred):")
print(cm)

## 12. Save evaluated results to Drive

In [ ]:
EVAL_PATH = os.path.join(RESULT_DIR, "haiku_experiment_A_zeroshot_densitydef_evaluated.csv")
haiku_A_zsdd_eval.to_csv(EVAL_PATH, index=False)

print("Saved:", len(haiku_A_zsdd_eval), "rows")
print("Path:", EVAL_PATH)